In [ ]:
##################### Phase 5: Clinical Decision Support #####################

import pandas as pd
import numpy as np

# 1. Define the Clinical Rules Engine Function
def calculate_clinical_risk(row):
    # Rule A: QC Failure
    if row['CY5'] == 0 and row['Total_HPV_Hits'] == 0:
        return "Invalid - Re-extract Sample"
    
    # Rule B: Highest Risk (HPV 16 or 18 Positive)
    if row['FAM'] == 1 or row['ROX'] == 1:
        return "High Risk - Refer to Colposcopy"
    
    # Rule C: Moderate Risk (Other High-Risk Genotypes Positive)
    if row['HEX'] == 1:
        return "Moderate Risk - Repeat in 1 Year"
    
    # Rule D: Negative
    if row['Total_HPV_Hits'] == 0 and row['CY5'] == 1:
        return "Routine Screening - Negative"
        
    return "Manual Review Required"

# 2. Apply the Rules Engine to our CLEANED Patient Dataset
patient_only_df['Clinical_Recommendation'] = patient_only_df.apply(calculate_clinical_risk, axis=1)

# 3. Summarize the Final Medical Actions
print("--- Clinical Decision Support Summary (Valid Patients Only) ---")
recommendation_counts = patient_only_df['Clinical_Recommendation'].value_counts().reset_index()
recommendation_counts.columns = ['Action / Recommendation', 'Number of Samples']
display(recommendation_counts)

# 4. Preview the High-Risk Patient Registry (Masked)
print("\n--- High-Risk Patient Registry (Preview) ---")
high_risk_patients = patient_only_df[patient_only_df['Clinical_Recommendation'] == "High Risk - Refer to Colposcopy"]
display(high_risk_patients[['Run_ID', 'Well', 'Total_HPV_Hits', 'Clinical_Recommendation']].head(10))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# 1. Merge the Cq values with the patient-level infection counts
hpv_cq_data = clean_df[(clean_df['Target'] != 'CY5') & (clean_df['Is_Positive'] == 1)].copy()
hpv_cq_data = hpv_cq_data.dropna(subset=['Cq_Value'])

# Merge with the CLEANED dataset (patient_only_df)
hpv_viral_loads = pd.merge(
    hpv_cq_data, 
    patient_only_df[['Run_ID', 'Well', 'Source_File', 'Total_HPV_Hits']], 
    on=['Run_ID', 'Well', 'Source_File'], 
    how='inner'
)

# 2. Categorize into Single vs Co-infections
clinical_viral_loads = hpv_viral_loads[hpv_viral_loads['Total_HPV_Hits'].isin([1, 2])].copy()

clinical_viral_loads['Infection_Type'] = clinical_viral_loads['Total_HPV_Hits'].map({
    1: 'Single Infection', 
    2: 'Co-infection (Double)'
})

# 3. Statistical Testing (Mann-Whitney U Test)
single_cq = clinical_viral_loads[clinical_viral_loads['Infection_Type'] == 'Single Infection']['Cq_Value']
co_cq = clinical_viral_loads[clinical_viral_loads['Infection_Type'] == 'Co-infection (Double)']['Cq_Value']

stat, p_value = stats.mannwhitneyu(single_cq, co_cq, alternative='two-sided')

print(f"--- Statistical Analysis Results ---")
print(f"Single Infection Median Cq: {single_cq.median():.2f}")
print(f"Co-infection Median Cq: {co_cq.median():.2f}")
print(f"Mann-Whitney U p-value: {p_value:.4f}\n")

if p_value < 0.05:
    print("Conclusion: There IS a statistically significant difference in viral loads between single and co-infections.")
else:
    print("Conclusion: There is NO statistically significant difference in viral loads between the groups.")

# 4. Visualization (Violin Plot with Swarm)
plt.figure(figsize=(10, 6))
sns.violinplot(data=clinical_viral_loads, x='Infection_Type', y='Cq_Value', inner='quartile')
sns.stripplot(data=clinical_viral_loads, x='Infection_Type', y='Cq_Value', color='black', alpha=0.4, size=4, jitter=True)

plt.title('Viral Load Dynamics: Single vs. Co-infections', fontsize=14, fontweight='bold')
plt.xlabel('Infection Complexity', fontsize=12)
plt.ylabel('Cq Value (Lower = Higher Viral Load)', fontsize=12)

plt.annotate(f'Mann-Whitney p-value: {p_value:.4f}', 
             xy=(0.5, 0.95), xycoords='axes fraction', 
             ha='center', va='center', fontsize=12, 
             bbox=dict(facecolor='white', alpha=0.9, edgecolor='gray', boxstyle='round,pad=0.5'))

plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()